# Grammar Error Correction Training Workflow

This notebook walks through the essential preparation steps for the grammar error correction project.

It covers:
- loading the cleaned dataset
- inspecting and preparing the data
- tokenizer analysis
- dataset splitting and tokenization
- model and trainer setup
- a non-training sanity check

The notebook is designed to be readable, reusable, and safe to run without starting model training.

## 1. Import Required Libraries

We use the standard scientific Python stack plus Hugging Face libraries for tokenizer and model preparation.

In [ ]:
# Core utilities
from __future__ import annotations

import random
import sys
from pathlib import Path

# Resolve the project root so the notebook can run from any launch location.
def find_project_root(start_path: Path | None = None) -> Path:
    """Find the repository root by walking upward until core project folders exist."""

    current_path = (start_path or Path.cwd()).resolve()
    for candidate in [current_path, *current_path.parents]:
        if (candidate / "model").exists() and (candidate / "datasets").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing model/ and datasets/.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Data handling and numerical computing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling utilities used in the notebook examples
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Project modules
from model.config import PipelineConfig, ModelConfig, DataConfig, TrainingConfig
from model.dataset import DatasetBuilderConfig, GrammarDatasetBuilder
from model.tokenizer_analysis import TokenizerAnalysisConfig, TokenizerAnalyzer
from model.tokenization import GrammarTokenizer, TokenizationConfig
from model.train import TrainingPreparationPipeline
from model.sanity_check import run_sanity_check

# Hugging Face / PyTorch components for model preparation
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

print(f"Project root resolved to: {PROJECT_ROOT}")

## 2. Configure Notebook Settings and Random Seed

Set display options and initialize randomness so the workflow is reproducible.

In [ ]:
# Notebook display settings
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Project paths
CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "processed" / "train_clean_cleaned.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned dataset: {CLEANED_DATASET_PATH}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Models directory: {MODELS_DIR}")

## 3. Load Data from Source

The notebook uses the cleaned sentence-pair CSV produced by the preprocessing stage.
The expected schema is `source` and `target`.

In [ ]:
def load_cleaned_dataset(dataset_path: Path) -> pd.DataFrame:
    """Load the cleaned grammar-correction dataset from CSV."""

    if not dataset_path.exists():
        raise FileNotFoundError(f"Cleaned dataset not found: {dataset_path}")

    return pd.read_csv(dataset_path)


dataframe = load_cleaned_dataset(CLEANED_DATASET_PATH)
print(f"Loaded rows: {len(dataframe)}")
print(dataframe.head(3))

## 4. Inspect Data Structure and Types

Review the dataset shape, column names, types, and missing values before making changes.

In [ ]:
print("Dataset shape:", dataframe.shape)
print("Columns:", list(dataframe.columns))
print("Data types:\n", dataframe.dtypes)
print("Missing values:\n", dataframe.isna().sum())
print("Empty source rows:", (dataframe["source"].astype(str).str.strip() == "").sum())
print("Empty target rows:", (dataframe["target"].astype(str).str.strip() == "").sum())

## 5. Clean and Transform Data

The preprocessing stage already produced a cleaned dataset, but this cell validates the final schema and prepares the data for downstream steps.

In [ ]:
def normalize_sentence_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply light validation and normalize whitespace for the notebook workflow."""

    normalized = frame.copy()
    normalized = normalized.dropna(subset=["source", "target"])
    normalized["source"] = normalized["source"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    normalized["target"] = normalized["target"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    normalized = normalized[(normalized["source"] != "") & (normalized["target"] != "")]
    normalized = normalized.drop_duplicates(subset=["source", "target"]).reset_index(drop=True)
    return normalized


clean_frame = normalize_sentence_frame(dataframe)
print(f"Rows after validation: {len(clean_frame)}")
print(clean_frame.head(3))

## 6. Perform Exploratory Data Analysis

Summarize sentence lengths, vocabulary, and changed versus unchanged pairs to understand the cleaned corpus.

In [ ]:
source_word_lengths = clean_frame["source"].str.split().str.len()
target_word_lengths = clean_frame["target"].str.split().str.len()

eda_summary = pd.DataFrame(
    {
        "source_word_length": source_word_lengths,
        "target_word_length": target_word_lengths,
        "sentence_changed": clean_frame["source"] != clean_frame["target"],
    }
)

print(eda_summary.describe(include="all"))
print("Changed pairs:", int(eda_summary["sentence_changed"].sum()))
print("Unchanged pairs:", int((~eda_summary["sentence_changed"]).sum()))
print("Source vocabulary size:", len({token.lower() for sentence in clean_frame["source"] for token in sentence.split()}))
print("Target vocabulary size:", len({token.lower() for sentence in clean_frame["target"] for token in sentence.split()}))

## 7. Create Visualizations

Visual inspection helps confirm the token-length distribution and the relationship between source and target sentence lengths.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(source_word_lengths, bins=40, kde=False, ax=axes[0], color="#1f77b4")
axes[0].set_title("Source Sentence Token Lengths")
axes[0].set_xlabel("Tokens")
axes[0].set_ylabel("Frequency")

sns.histplot(target_word_lengths, bins=40, kde=False, ax=axes[1], color="#ff7f0e")
axes[1].set_title("Target Sentence Token Lengths")
axes[1].set_xlabel("Tokens")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
sns.scatterplot(x=source_word_lengths, y=target_word_lengths, alpha=0.3, s=15)
plt.title("Source vs Target Token Lengths")
plt.xlabel("Source tokens")
plt.ylabel("Target tokens")
plt.show()

## 8. Engineer and Select Features

Prepare a prompt-style input column and keep the target sentence as the label for the sequence-to-sequence workflow.

In [ ]:
feature_frame = clean_frame.copy()
feature_frame["prompt"] = "fix grammar: " + feature_frame["source"]
feature_frame = feature_frame[["prompt", "source", "target"]]

print(feature_frame.head(3))
print("Feature frame shape:", feature_frame.shape)

## 9. Train a Baseline Model

This notebook prepares the model and trainer objects, but it does not call `trainer.train()`.
That keeps the workflow safe to run while still validating the full training setup.

In [ ]:
pipeline_config = PipelineConfig(
    model=ModelConfig(),
    data=DataConfig(cleaned_dataset_path=CLEANED_DATASET_PATH, max_input_length=64, max_target_length=64),
    training=TrainingConfig(batch_size=4, learning_rate=5e-5, epochs=1, output_dir=MODELS_DIR / "flan_t5_small_gec", logging_dir=REPORTS_DIR / "training_logs"),
)

analysis_result = TokenizerAnalyzer(
    TokenizerAnalysisConfig(cleaned_dataset_path=CLEANED_DATASET_PATH)
).analyze()

print("Recommended max input length:", analysis_result.recommended_max_input_length)

# Build the dataset and tokenize it using the project modules.
dataset_builder = GrammarDatasetBuilder(DatasetBuilderConfig(cleaned_dataset_path=CLEANED_DATASET_PATH))
dataset_dict = dataset_builder.build()
tokenizer_config = TokenizationConfig(
    max_input_length=analysis_result.recommended_max_input_length,
    max_target_length=analysis_result.recommended_max_input_length,
)
tokenized_dataset = GrammarTokenizer(tokenizer_config).tokenize_dataset(dataset_dict)

# Load the model and trainer objects, but do not train.
tokenizer = AutoTokenizer.from_pretrained(pipeline_config.model.tokenizer_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(pipeline_config.model.model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
training_arguments = Seq2SeqTrainingArguments(
    output_dir=str(pipeline_config.training.output_dir),
    logging_dir=str(pipeline_config.training.logging_dir),
    per_device_train_batch_size=pipeline_config.training.batch_size,
    per_device_eval_batch_size=pipeline_config.training.batch_size,
    learning_rate=pipeline_config.training.learning_rate,
    num_train_epochs=pipeline_config.training.epochs,
    weight_decay=pipeline_config.training.weight_decay,
    gradient_accumulation_steps=pipeline_config.training.gradient_accumulation_steps,
    fp16=pipeline_config.training.fp16,
    seed=pipeline_config.training.random_seed,
    logging_steps=pipeline_config.training.logging_steps,
    save_strategy=pipeline_config.training.save_strategy,
    eval_strategy=pipeline_config.training.evaluation_strategy,
    save_total_limit=pipeline_config.training.save_total_limit,
    report_to=[],
    predict_with_generate=False,
    remove_unused_columns=False,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Train samples:", len(dataset_dict["train"]))
print("Validation samples:", len(dataset_dict["validation"]))
print("Test samples:", len(dataset_dict["test"]))
print("Vocabulary size:", tokenizer.vocab_size)
print("Model parameters:", sum(parameter.numel() for parameter in model.parameters()))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
print("Trainer type:", type(trainer).__name__)

## 10. Evaluate Model Performance

Run a single forward pass on a small batch and inspect the loss value without training the model.

In [ ]:
sanity_status = run_sanity_check(pipeline_config, sample_count=5)
print("Sanity check exit code:", sanity_status)

print("Note: this notebook does not call trainer.train(). It validates the full pipeline only.")

## 11. Save Outputs and Document Findings

Document the files generated by the workflow and summarize the current readiness state for future training.

In [ ]:
print("Artifacts produced by the workflow:")
print(f"- Tokenizer report: {REPORTS_DIR / 'tokenizer_report.txt'}")
print(f"- Tokenizer histogram: {REPORTS_DIR / 'token_length_histogram.png'}")
print(f"- Cleaned dataset: {CLEANED_DATASET_PATH}")
print(f"- Model output directory: {pipeline_config.training.output_dir}")
print("\nFindings:")
print("- The cleaned dataset is ready for seq2seq preparation.")
print("- The tokenizer analysis provides a recommended max_input_length.")
print("- The trainer is instantiated but no training step is executed.")
print("- The sanity check validates a forward pass and confirms readiness for training.")